In [17]:
!pip install imutils


In [21]:
import os

# Шлях до репо
LPRNET_REPO_PATH = 'LPRNet_Pytorch'

# Повертаємося в корінь проекту (про всяк випадок)
while not os.path.exists(LPRNET_REPO_PATH) and os.getcwd() != '/':
    os.chdir('..')

print(f"Поточна папка: {os.getcwd()}")

# Відкат змін git
try:
    os.chdir(LPRNET_REPO_PATH)
    !git checkout .
    print("Репозиторій відновлено до оригінального стану.")
except Exception as e:
    print(f"Помилка відновлення (можливо, це не git репо?): {e}")
finally:
    # Повертаємося назад у корінь проекту
    os.chdir('..')

Поточна папка: /mnt/c/Users/user/PycharmProjects/Diploma/try5
Updated 2 paths from the index
Репозиторій відновлено до оригінального стану.


In [22]:
import sys
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time


LPRNET_REPO_PATH = 'LPRNet_Pytorch'
IMG_SIZE = (94, 24)
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.001
LPR_MAX_LEN = 18

CHARS = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '.']
CHARS_DICT = {char: i for i, char in enumerate(CHARS)}
NUM_CLASS = len(CHARS)

TRAIN_CROPS_DIR = 'lpr_datasets/lprnet/crops'

if LPRNET_REPO_PATH not in sys.path:
    sys.path.append(LPRNET_REPO_PATH)

try:
    from model.LPRNet import LPRNet
    print("Модель LPRNet успішно імпортовано.")
except ImportError:
    print("Помилка: Не можу знайти model/LPRNet.py.")

class CustomLPRDataset(Dataset):
    def __init__(self, img_dir, img_size, chars_dict):
        self.img_dir = img_dir
        self.img_paths = []
        self.labels = []
        self.img_size = img_size
        self.chars_dict = chars_dict

        files = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
        print(f"Знайдено файлів у {img_dir}: {len(files)}")

        for filename in files:
            try:
                base = os.path.splitext(filename)[0]
                if '_' in base:
                    label = base.split('_')[-1]
                else:
                    label = base

                if all(c in chars_dict for c in label):
                    self.img_paths.append(os.path.join(img_dir, filename))
                    self.labels.append(label)
            except Exception:
                continue

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, index):
        path = self.img_paths[index]
        label = self.labels[index]

        image = cv2.imread(path)
        if image is None:
             image = np.zeros((self.img_size[1], self.img_size[0], 3), dtype=np.uint8)

        image = cv2.resize(image, self.img_size)
        image = image.astype('float32')
        image -= 127.5
        image *= 0.0078125
        image = np.transpose(image, (2, 0, 1))

        return torch.from_numpy(image), label

def collate_fn(batch):
    imgs = []
    labels = []
    lengths = []
    for _, (img, label) in enumerate(batch):
        imgs.append(img)
        labels.extend([CHARS_DICT[c] for c in label])
        lengths.append(len(label))
    labels = np.asarray(labels).flatten().astype(np.int32)
    return torch.stack(imgs), torch.from_numpy(labels), lengths

def sparse_tuple_for_ctc(T_length, lengths):
    input_lengths = []
    target_lengths = []
    for l in lengths:
        target_lengths.append(l)
        input_lengths.append(T_length)
    return tuple(input_lengths), tuple(target_lengths)

def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Використовуємо пристрій: {device}")

    dataset = CustomLPRDataset(TRAIN_CROPS_DIR, IMG_SIZE, CHARS_DICT)

    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

    lprnet = LPRNet(lpr_max_len=LPR_MAX_LEN,
                    phase=True,
                    class_num=len(CHARS)+1,
                    dropout_rate=0.5).to(device)

    criterion = nn.CTCLoss(blank=len(CHARS), reduction='mean')
    optimizer = torch.optim.Adam(lprnet.parameters(), lr=LEARNING_RATE)

    print("--- Починаємо навчання ---")

    for epoch in range(EPOCHS):
        lprnet.train()
        epoch_loss = 0
        start_time = time.time()

        for images, labels, lengths in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = lprnet(images) # [Batch, Class, Time]
            log_probs = logits.permute(2, 0, 1) # [Time, Batch, Class]
            log_probs = log_probs.log_softmax(2).requires_grad_()

            T = log_probs.shape[0]

            input_lengths, target_lengths = sparse_tuple_for_ctc(T, lengths)

            try:
                loss = criterion(log_probs, labels, input_lengths=input_lengths, target_lengths=target_lengths)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            except Exception as e:
                print(f"Помилка батчу: {e}")
                continue

        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Time: {time.time()-start_time:.1f}s")

        if (epoch + 1) % 5 == 0:
            os.makedirs('weights', exist_ok=True)
            torch.save(lprnet.state_dict(), f'weights/lprnet_epoch_{epoch+1}.pth')

    torch.save(lprnet.state_dict(), 'weights/lprnet_best.pth')
    print("Навчання завершено! Фінальні ваги: weights/lprnet_best.pth")

if __name__ == '__main__':
    run_training()

Модель LPRNet успішно імпортовано.
Використовуємо пристрій: cuda
Знайдено файлів у lpr_datasets/lprnet/crops: 15656
--- Починаємо навчання ---
Epoch 1/5 | Loss: 1.2271 | Time: 129.2s
Epoch 2/5 | Loss: 0.2216 | Time: 108.4s
Epoch 3/5 | Loss: 0.1241 | Time: 109.0s
Epoch 4/5 | Loss: 0.0922 | Time: 110.5s
Epoch 5/5 | Loss: 0.0922 | Time: 111.3s
Навчання завершено! Фінальні ваги: weights/lprnet_best.pth
